# M1 — variantes tabulares con MLflow

Compara **arquitecturas** (LightGBM, XGBoost, RandomForest, LogReg) × **sets de features** (F1 solo categóricas, F2 + temporales, F3 + stats del mensaje) sobre la misma tarea que M1.

Protocolo idéntico al de `experiments/m1_tabular/train.py` y M2 o no vale comparar: **split temporal 80/20**, métricas **F1-macro + PR-AUC + recall malicious @1% FPR**. Todo queda en MLflow (`experiments/mlruns`).

Corre: `uv run jupyter nbconvert --to notebook --execute notebooks/m1_variants_mlflow.ipynb --output m1_variants_mlflow_out.ipynb --ExecutePreprocessor.timeout=1200`

In [1]:
import time
from pathlib import Path

import mlflow
import pandas as pd

p = Path.cwd()
ROOT = p if (p / "pyproject.toml").exists() else p.parent
print("ROOT:", ROOT)
mlflow.set_tracking_uri("file:" + str(ROOT / "experiments" / "mlruns"))
mlflow.set_experiment("m1-tabular-variants")

CATS = ["message_type", "stream_name", "product_name", "vendor_name", "pipeline",
        "org_id", "action", "severity", "protocol"]
BASE_NUMS = ["hour", "msg_len", "lag_h"]
TXT_STATS = ["word_count", "digit_ratio", "upper_ratio"]
FSETS = {"F1": CATS, "F2": CATS + BASE_NUMS, "F3": CATS + BASE_NUMS + TXT_STATS}

2026/10/23 00:06:07 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


ROOT: C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto


In [2]:
def features(df):
    df = df.copy()
    msg = df["message_sanitized"].fillna("")
    ts = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["hour"] = ts.dt.hour
    df["msg_len"] = msg.str.len()
    df["lag_h"] = ((df["timestamp"] - df["event_time"]) / 3600).clip(-1e4, 1e4).fillna(-1)
    df["word_count"] = msg.str.split().str.len().fillna(0)
    df["digit_ratio"] = msg.str.count(r"\d") / df["msg_len"].clip(lower=1)
    df["upper_ratio"] = msg.str.count(r"[A-Z]") / df["msg_len"].clip(lower=1)
    for c in CATS:
        df[c] = df[c].fillna("").astype(str)
    return df


def load_sample(n=60000):
    cols = ["label_binary", "timestamp", "event_time", "message_sanitized"] + CATS
    df = pd.read_parquet(ROOT / "data" / "signals" / "signals.parquet", columns=cols)
    df = features(df)
    mal = df[df.label_binary == "malicious"]
    rest = df[df.label_binary != "malicious"]
    n_rest = max(n - len(mal), 2000)
    sus = rest[rest.label_binary == "suspicious"]
    ben = rest[rest.label_binary == "benign"]
    n_sus = min(len(sus), n_rest // 3)
    out = pd.concat([mal, sus.sample(n_sus, random_state=42),
                     ben.sample(min(len(ben), n_rest - n_sus), random_state=42)])
    return out.sample(frac=1.0, random_state=42).reset_index(drop=True)


df = load_sample()
print(df.shape, df["label_binary"].value_counts().to_dict())
cutoff = df["timestamp"].quantile(0.8)
tr, te = df["timestamp"] <= cutoff, df["timestamp"] > cutoff
print("train/test:", int(tr.sum()), int(te.sum()))

(60000, 19) {'benign': 34848, 'suspicious': 17424, 'malicious': 7728}
train/test: 48000 12000


In [3]:
from scipy.sparse import csr_matrix
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, classification_report,
                             f1_score, roc_curve)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

le = LabelEncoder().fit(sorted(df["label_binary"].unique().tolist()))
y = le.transform(df["label_binary"])
print("clases:", dict(enumerate(le.classes_)))


def make_pre(fset):
    nums = [c for c in FSETS[fset] if c not in CATS]
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", max_categories=30), CATS),
        ("num", "passthrough", nums),
    ])


def builders():
    return {
        "lgbm": lambda: LGBMClassifier(n_estimators=200, learning_rate=0.08, num_leaves=63,
                                         class_weight="balanced", n_jobs=-1, verbose=-1),
        "xgb": lambda: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08,
                                       subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                       objective="multi:softprob", num_class=3,
                                       eval_metric="mlogloss", n_jobs=-1, random_state=42),
        "rf": lambda: RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                                               n_jobs=-1, random_state=42),
        "logreg": lambda: LogisticRegression(max_iter=500, C=0.5, class_weight="balanced"),
    }


def sample_weight(ytr):
    import numpy as np
    import pandas as pd
    s = pd.Series(ytr).value_counts()
    w = len(ytr) / (len(s) * s)
    return pd.Series(ytr).map(w).to_numpy()


def recall_at_fpr(y_true, scores, fpr_max=0.01):
    fpr, tpr, _ = roc_curve(y_true, scores)
    ok = fpr <= fpr_max
    return round(float(tpr[ok].max()) if ok.any() else 0.0, 4)

clases: {0: np.str_('benign'), 1: np.str_('malicious'), 2: np.str_('suspicious')}


In [4]:
GRID = [("lgbm", "F1"), ("lgbm", "F2"), ("lgbm", "F3"),
        ("xgb", "F2"), ("xgb", "F3"),
        ("rf", "F3"), ("logreg", "F2"), ("logreg", "F3")]
print(len(GRID), "corridas")
for name, fs in GRID:
    t0 = time.time()
    pre = make_pre(fs)
    Xtr = pre.fit_transform(df[tr][FSETS[fs]])
    Xte = pre.transform(df[te][FSETS[fs]])
    if not hasattr(Xtr, "toarray"):
        import numpy as np
        Xtr, Xte = np.asarray(Xtr), np.asarray(Xte)
    clf = builders()[name]()
    fit_kw = {"sample_weight": sample_weight(y[tr])} if name in ("xgb",) else {}
    if name in ("lgbm", "rf", "logreg"):
        pass  # class_weight ya balancea; xgb usa sample_weight (multi:softprob no acepta scale_pos_weight)
    clf.fit(Xtr, y[tr], **fit_kw)
    p = clf.predict(Xte)
    proba = clf.predict_proba(Xte)
    plabel = le.inverse_transform(p)
    yte_s = le.inverse_transform(y[te])
    rep = classification_report(yte_s, plabel, output_dict=True, zero_division=0)
    f1m = f1_score(yte_s, plabel, average="macro")
    with mlflow.start_run(run_name=f"{name}-{fs}"):
        mlflow.log_params({"model": name, "fset": fs, "n": len(df),
                           "n_train": int(tr.sum()), "n_test": int(te.sum())})
        mlflow.log_metric("f1_macro", round(float(f1m), 4))
        for i, c in enumerate(le.classes_):
            mlflow.log_metric(f"f1_{c}", round(rep[c]["f1-score"], 4))
            mlflow.log_metric(f"recall_{c}", round(rep[c]["recall"], 4))
            mlflow.log_metric(f"precision_{c}", round(rep[c]["precision"], 4))
            try:
                mlflow.log_metric(f"pr_auc_{c}", round(float(average_precision_score((y[te] == i).astype(int), proba[:, i])), 4))
            except Exception:
                mlflow.log_metric(f"pr_auc_{c}", 0.0)
        mal_i = list(le.classes_).index("malicious")
        mlflow.log_metric("recall_mal_at_1fpr",
                          recall_at_fpr((y[te] == mal_i).astype(int), proba[:, mal_i]))
        mlflow.log_metric("train_s", round(time.time() - t0, 1))
    print(f"{name}-{fs}: F1-macro={f1m:.4f} mal-recall={rep['malicious']['recall']:.4f} ({time.time()-t0:.0f}s)")

8 corridas


lgbm-F1: F1-macro=0.5951 mal-recall=1.0000 (16s)


lgbm-F2: F1-macro=0.5052 mal-recall=0.0000 (21s)


lgbm-F3: F1-macro=0.5092 mal-recall=0.0183 (17s)


xgb-F2: F1-macro=0.5131 mal-recall=0.0112 (11s)


...

rf-F3: F1-macro=0.5217 mal-recall=0.0286 (25s)


C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logreg-F2: F1-macro=0.6148 mal-recall=0.9679 (23s)


C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logreg-F3: F1-macro=0.6095 mal-recall=0.9798 (21s)


In [5]:
runs = mlflow.search_runs(experiment_ids=[mlflow.get_experiment_by_name("m1-tabular-variants").experiment_id])
cols = ["params.model", "params.fset", "metrics.f1_macro", "metrics.recall_malicious",
        "metrics.pr_auc_malicious", "metrics.recall_mal_at_1fpr", "metrics.precision_malicious",
        "metrics.f1_suspicious", "metrics.precision_suspicious",
        "metrics.train_s"]
board = runs[cols].sort_values("metrics.f1_macro", ascending=False).reset_index(drop=True)
board
board.to_csv(ROOT / "experiments" / "m1_tabular" / "leaderboard.csv", index=False)
print("guardado experiments/m1_tabular/leaderboard.csv")
print("baseline m1.pkl: F1-macro=0.5991 mal-recall=0.0045 -> ¿alguna variante lo supera?")

guardado experiments/m1_tabular/leaderboard.csv
baseline m1.pkl: F1-macro=0.5991 mal-recall=0.0045 -> ¿alguna variante lo supera?


## Lectura

- Si una variante supera F1-macro **0.5991** con mejor recall malicious, se promueve a `models/m1.pkl` (mismo formato) y el paper gana una fila en la tabla M1.
- `F3 - F2` mide cuánto aporta el texto crudo sin TF-IDF; si ≈0, el TF-IDF de M2 está justificado.
- `mlflow ui --backend-store-uri experiments/mlruns` para ver curvas y comparar.